# CYA Colab setup

Run this notebook through the official VS Code Colab extension. It mounts Drive, prepares the disposable runtime checkout, installs dependencies without replacing Colab's PyTorch/CUDA build, optionally stages data under `/content`, and runs the strict smoke check.

In [6]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")
DRIVE_FOLDER_ID = "1c-IVvAiHlApA49CtU3QQH9XqQDmkbO8U"
DRIVE_FOLDER_URL = f"https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID}"
DRIVE_DATA_ROOT = DRIVE_ROOT / "hackathon_data"
DRIVE_ARTIFACT_ROOT = DRIVE_ROOT / "cya-techjam26/artifacts"
LOCAL_PROJECT_ROOT = Path("/content/cya-techjam26")
LOCAL_DATA_ROOT = Path("/content/hackathon_data")
REPOSITORY_URL = "https://github.com/maxi-cmyk/cya-techjam26.git"

assert DRIVE_DATA_ROOT.exists(), (
    f"Drive dataset not found: {DRIVE_DATA_ROOT}. "
    f"Open {DRIVE_FOLDER_URL}, add a shortcut to My Drive, and name it hackathon_data."
)
DRIVE_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Drive folder ID: {DRIVE_FOLDER_ID}")
print(f"Drive data: {DRIVE_DATA_ROOT}")
print(f"Persistent artifacts: {DRIVE_ARTIFACT_ROOT}")

Drive folder ID: 1c-IVvAiHlApA49CtU3QQH9XqQDmkbO8U
Drive data: /content/drive/MyDrive/hackathon_data
Persistent artifacts: /content/drive/MyDrive/cya-techjam26/artifacts


In [8]:
import subprocess

if (LOCAL_PROJECT_ROOT / ".git").exists():
    subprocess.run(
        ["git", "pull", "--ff-only"],
        cwd=LOCAL_PROJECT_ROOT,
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", REPOSITORY_URL, str(LOCAL_PROJECT_ROOT)],
        check=True,
    )

In [9]:
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", "requirements-colab.txt"],
    cwd=LOCAL_PROJECT_ROOT,
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps"],
    cwd=LOCAL_PROJECT_ROOT,
    check=True,
)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-e', '.', '--no-deps'], returncode=0)

## Optional local data staging

Keep `COPY_DATA = False` until `DATA_SUBDIRECTORY` matches the Drive folder needed for the current task. Copying can take time, but training from `/content` avoids repeated small-file reads through the Drive mount.

In [ ]:
import shutil

COPY_DATA = True
DATA_SUBDIRECTORY = Path("raw/sid_set")  # Change to raw/sid_set for Task 2.

source_data = DRIVE_DATA_ROOT / DATA_SUBDIRECTORY
local_data = LOCAL_DATA_ROOT / DATA_SUBDIRECTORY
assert source_data.exists(), f"Dataset subset not found: {source_data}"

if COPY_DATA:
    shutil.copytree(source_data, local_data, dirs_exist_ok=True)
    print(f"Staged data at {local_data}")
else:
    print(f"Ready to copy {source_data} to {local_data}")

In [ ]:
subprocess.run(
    [sys.executable, "scripts/smoke_check.py", "--config", "configs/colab.json"],
    cwd=LOCAL_PROJECT_ROOT,
    check=True,
)

## Sync durable outputs

Run this after a completed experiment. It copies rather than moves, so an interrupted Drive operation does not remove the local output.

In [ ]:
local_artifacts = LOCAL_PROJECT_ROOT / "artifacts"
if local_artifacts.exists():
    shutil.copytree(local_artifacts, DRIVE_ARTIFACT_ROOT, dirs_exist_ok=True)
    print(f"Copied artifacts to {DRIVE_ARTIFACT_ROOT}")
else:
    print("No local artifacts to sync yet.")

In [ ]:
from pathlib import Path
import subprocess

source = Path("/content/drive/MyDrive/hackathon_data/raw/sid_set")
destination = Path("/content/hackathon_data/raw/sid_set")

assert source.is_dir(), f"Drive source not found: {source}"
destination.mkdir(parents=True, exist_ok=True)

subprocess.run(
    ["git", "pull", "--ff-only"],
    cwd="/content/cya-techjam26",
    check=True,
)

images = destination / "images"
labels = destination / "labels.csv"

assert images.is_dir(), f"Images folder missing: {images}"
assert labels.is_file(), f"Labels file missing: {labels}"

image_count = sum(1 for path in images.iterdir() if path.is_file())

print("\nVerification")
print("Images:", image_count)
print("Labels file:", labels)
print("Dataset ready:", image_count == 20_000)

assert image_count == 20_000, f"Expected 20,000 images, found {image_count}"
print("\nPASS: raw SID dataset is ready.")


Verification
Images: 20000
Labels file: /content/hackathon_data/raw/sid_set/labels.csv
Dataset ready: True

PASS: raw SID dataset is ready.


In [ ]:
from pathlib import Path

destination = Path("/content/hackathon_data/raw/sid_set")
images = destination / "images"
labels = destination / "labels.csv"

print("Images directory exists:", images.is_dir())
print("Labels file exists:", labels.is_file())

image_count = (
    sum(1 for path in images.iterdir() if path.is_file())
    if images.is_dir()
    else 0
)

print("Copied images:", image_count)
print("Expected images:", 20_000)
print("Missing images:", 20_000 - image_count)
print("Dataset complete:", image_count == 20_000 and labels.is_file())

Images directory exists: True
Labels file exists: True
Copied images: 20000
Expected images: 20000
Missing images: 0
Dataset complete: True


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import shutil

source_root = Path("/content/drive/MyDrive/hackathon_data/raw/sid_set")
source_images = source_root / "images"

destination_root = Path("/content/hackathon_data/raw/sid_set")
destination_images = destination_root / "images"
destination_images.mkdir(parents=True, exist_ok=True)

# Ensure labels are present.
shutil.copy2(source_root / "labels.csv", destination_root / "labels.csv")

source_files = [path for path in source_images.iterdir() if path.is_file()]

pending = []
for source_file in source_files:
    destination_file = destination_images / source_file.name

    if (
        not destination_file.exists()
        or destination_file.stat().st_size != source_file.stat().st_size
    ):
        pending.append(source_file)

print("Total source files:", len(source_files))
print("Already complete:", len(source_files) - len(pending))
print("Remaining:", len(pending))


def copy_one(source_file):
    destination_file = destination_images / source_file.name
    temporary_file = destination_images / f"{source_file.name}.part"

    shutil.copy2(source_file, temporary_file)
    temporary_file.replace(destination_file)

    return source_file.name


errors = []

with ThreadPoolExecutor(max_workers=8) as executor:
    futures = {
        executor.submit(copy_one, source_file): source_file
        for source_file in pending
    }

    for completed, future in enumerate(as_completed(futures), start=1):
        try:
            future.result()
        except Exception as error:
            errors.append((futures[future].name, str(error)))

        if completed % 100 == 0 or completed == len(futures):
            print(
                f"Progress: {completed}/{len(futures)} remaining files processed; "
                f"errors: {len(errors)}"
            )

final_count = sum(
    1 for path in destination_images.iterdir()
    if path.is_file() and not path.name.endswith(".part")
)

print("\nFinal verification")
print("Images:", final_count)
print("Errors:", len(errors))
print("Dataset complete:", final_count == 20_000)

if errors:
    print("First errors:", errors[:10])

assert not errors, f"{len(errors)} files failed to copy"
assert final_count == 20_000, f"Expected 20,000 images, found {final_count}"

print("PASS: raw SID dataset is ready.")

Total source files: 20000
Already complete: 7646
Remaining: 12354
Progress: 100/12354 remaining files processed; errors: 0
Progress: 200/12354 remaining files processed; errors: 0
Progress: 300/12354 remaining files processed; errors: 0
Progress: 400/12354 remaining files processed; errors: 0
Progress: 500/12354 remaining files processed; errors: 0
Progress: 600/12354 remaining files processed; errors: 0
Progress: 700/12354 remaining files processed; errors: 0
Progress: 800/12354 remaining files processed; errors: 0
Progress: 900/12354 remaining files processed; errors: 0
Progress: 1000/12354 remaining files processed; errors: 0
Progress: 1100/12354 remaining files processed; errors: 0
Progress: 1200/12354 remaining files processed; errors: 0
Progress: 1300/12354 remaining files processed; errors: 0
Progress: 1400/12354 remaining files processed; errors: 0
Progress: 1500/12354 remaining files processed; errors: 0
Progress: 1600/12354 remaining files processed; errors: 0
Progress: 1700/